In [5]:
import numpy as np
import pandas as pd

In [6]:
df = pd.read_csv("D:\Data Analytics\Projects\Anti-Money Laundering (AML) transaction monitoring\HI-Small_Trans.csv")
df.head(5)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,01-09-2022 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,01-09-2022 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,01-09-2022 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,01-09-2022 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,01-09-2022 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


**Phase 1 — Understanding data**

How many rows and columns are there, and what are the data types?

In [3]:
df.shape

(1048575, 11)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   Timestamp           1048575 non-null  object 
 1   From Bank           1048575 non-null  int64  
 2   Account             1048575 non-null  object 
 3   To Bank             1048575 non-null  int64  
 4   Account.1           1048575 non-null  object 
 5   Amount Received     1048575 non-null  float64
 6   Receiving Currency  1048575 non-null  object 
 7   Amount Paid         1048575 non-null  float64
 8   Payment Currency    1048575 non-null  object 
 9   Payment Format      1048575 non-null  object 
 10  Is Laundering       1048575 non-null  int64  
dtypes: float64(2), int64(3), object(6)
memory usage: 88.0+ MB


In [9]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   Timestamp           1048575 non-null  datetime64[ns]
 1   From Bank           1048575 non-null  int64         
 2   Account             1048575 non-null  object        
 3   To Bank             1048575 non-null  int64         
 4   Account.1           1048575 non-null  object        
 5   Amount Received     1048575 non-null  float64       
 6   Receiving Currency  1048575 non-null  object        
 7   Amount Paid         1048575 non-null  float64       
 8   Payment Currency    1048575 non-null  object        
 9   Payment Format      1048575 non-null  object        
 10  Is Laundering       1048575 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(3), object(5)
memory usage: 88.0+ MB


Are there missing values, duplicate records, or invalid values?

In [5]:
df.isnull().sum()

Timestamp             0
From Bank             0
Account               0
To Bank               0
Account.1             0
Amount Received       0
Receiving Currency    0
Amount Paid           0
Payment Currency      0
Payment Format        0
Is Laundering         0
dtype: int64

In [6]:
df[df.duplicated]

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
863295,01-09-2022 16:20,12004,800C927C1,12004,800C927C0,0.000008,Bitcoin,0.080000,Euro,ACH,0
863296,01-09-2022 16:20,12004,800C927C1,220,813D8C1E1,0.000008,Bitcoin,0.000008,Bitcoin,Bitcoin,0


In [11]:
df = df.drop_duplicates()

In [12]:
df[df.duplicated]

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering


What is the date range covered by the transactions?

In [14]:
start_dt = df['Timestamp'].min()
end_dt = df['Timestamp'].max()

print(f'Date range is : {start_dt} to {end_dt}')

Date range is : 2022-01-09 00:00:00 to 2022-09-09 08:29:00


What percentage of transactions are labelled laundering versus non-laundering?

In [25]:
laundering_count = (df['Is Laundering'] == 1).sum()
non_laundering_count = (df['Is Laundering'] == 0).sum()
total_transactions = len(df)

laundering_per = (laundering_count / total_transactions) * 100
non_laundering_per = (non_laundering_count / total_transactions) * 100

print(f"Laundering percentage: {laundering_per:.4f}%")
print(f"Non-laundering percentage: {non_laundering_per:.4f}%")

Laundering percentage: 0.0587%
Non-laundering percentage: 99.9413%


**Phase 2 — Explore transaction patterns**

What do transaction amounts look like? Are there extreme values or outliers?

In [5]:
df[['Amount Received', 'Amount Paid']]

,Amount Received,Amount Paid
0,3697.34,3697.34
1,0.01,0.01
2,14675.57,14675.57
3,2806.97,2806.97
4,36682.97,36682.97
...,...,...
1048570,27186.66,27186.66
1048571,132355.38,132355.38
1048572,3764.12,3764.12
1048573,892.20,892.20


In [21]:
df[['Amount Received', 'Amount Paid']].describe().style.format("{:,.2f}")

,Amount Received,Amount Paid
count,"1,048,575.00","1,048,575.00"
mean,"7,137,529.80","5,917,423.87"
std,"602,999,068.49","469,306,005.56"
min,0.00,0.00
25%,101.19,101.92
50%,"1,925.45","1,928.91"
75%,"24,768.46","24,693.39"
max,"284,000,000,000.00","284,000,000,000.00"


*Finding outliers*

In [7]:
amount_paid_q1 = df['Amount Paid'].quantile(0.25)
amount_paid_q3 = df['Amount Paid'].quantile(0.75)
print('Amount paid')
print(f'Q1: {amount_paid_q1} \nQ2: {amount_paid_q3}')
amount_recieved_q1 = df['Amount Received'].quantile(0.25)
amount_recieved_q3 = df['Amount Received'].quantile(0.75)
print('Amount recieved')
print(f'Q1: {amount_recieved_q1} \nQ2: {amount_recieved_q3}')

Amount paid
Q1: 101.92 
Q2: 24693.39
Amount recieved
Q1: 101.195 
Q2: 24768.46


In [8]:
amount_paid_iqr = amount_paid_q3-amount_paid_q1
amount_recieved_iqr = amount_recieved_q3-amount_recieved_q1

print(f'IQR of Amount paid: {amount_paid_iqr}')
print(f'IQR of Amount recieved: {amount_recieved_iqr}')

IQR of Amount paid: 24591.47
IQR of Amount recieved: 24667.265


In [9]:
'''Lower bound = Q1 - 1.5 x IQR*
Upper bound = Q3 + 1.5 x IQR'''

amount_paid_lb = amount_paid_q1-1.5*amount_paid_iqr
amount_paid_ub = amount_paid_q3+1.5*amount_paid_iqr
print('Amount paid')
print(f'Lower bound: {amount_paid_lb} \nUpper bound{amount_paid_ub}')

amount_recieved_lb = amount_recieved_q1-1.5*amount_recieved_iqr
amount_recieved_ub = amount_recieved_q3+1.5*amount_recieved_iqr
print('Amount recieved')
print(f'Lower bound: {amount_recieved_lb} \nUpper bound{amount_recieved_ub}')

Amount paid
Lower bound: -36785.285 
Upper bound61580.595
Amount recieved
Lower bound: -36899.7025 
Upper bound61769.3575


In [13]:
outlier_count1 = (df['Amount Received']>amount_recieved_ub).sum()
print(f'Outliers for the column "Amount Recieved" are: {outlier_count1}')

outlier_count2 = (df['Amount Paid']>amount_paid_ub).sum()
print(f'Outliers for the column "Amount Paid" are: {outlier_count2}')

Outliers for the column "Amount Recieved" are: 197322
Outliers for the column "Amount Paid" are: 197081


In [14]:
total_transactions = len(df)

print(f"Amount Received: {outlier_count1 / total_transactions * 100:.2f}%")
print(f"Amount Paid: {outlier_count2 / total_transactions * 100:.2f}%")

Amount Received: 18.82%
Amount Paid: 18.80%
